In [1]:
!pip install torch torchvision tqdm

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.models import mobilenet_v2

from tqdm import tqdm

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [9]:
dataset_path ="/kaggle/input/datasets/sudhan44/gender-classification/Dataset_Gender"

In [39]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [40]:
train_data = datasets.ImageFolder(dataset_path+"/train", transform=train_transform)

valid_data = datasets.ImageFolder(dataset_path+"/valid", transform=test_transform)

test_data = datasets.ImageFolder(dataset_path+"/test", transform=test_transform)

print("Classes:", train_data.classes)

Classes: ['Female', 'Male']


In [42]:
train_loader = DataLoader(
    train_data,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_data,
    batch_size=32,
    num_workers=2
)

test_loader = DataLoader(
    test_data,
    batch_size=32,
    num_workers=2
)

In [45]:
model = mobilenet_v2(weights="DEFAULT")

In [46]:
for param in model.features.parameters():
    param.requires_grad = False

In [47]:
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)

In [48]:
model = model.to(device)

In [49]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

In [50]:
epochs = 5

for epoch in range(epochs):

    model.train()

    correct = 0
    total = 0

    for images, labels in tqdm(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    acc = 100 * correct / total

    print(f"Epoch {epoch+1} Accuracy:{acc:.2f}%")

100%|██████████| 219/219 [00:15<00:00, 13.69it/s]


Epoch 1 Accuracy:84.93%


100%|██████████| 219/219 [00:15<00:00, 14.13it/s]


Epoch 2 Accuracy:89.14%


100%|██████████| 219/219 [00:15<00:00, 14.02it/s]


Epoch 3 Accuracy:89.94%


100%|██████████| 219/219 [00:15<00:00, 14.14it/s]


Epoch 4 Accuracy:90.07%


100%|██████████| 219/219 [00:15<00:00, 14.03it/s]

Epoch 5 Accuracy:90.44%


In [51]:
for param in model.features[14:].parameters():
    param.requires_grad = True

In [52]:
optimizer = optim.Adam(model.parameters(), lr=0.00001)

In [53]:
epochs = 5

for epoch in range(epochs):

    model.train()

    correct = 0
    total = 0

    for images, labels in tqdm(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    acc = 100 * correct / total

    print(f"FineTune Epoch {epoch+1} Accuracy:{acc:.2f}%")

100%|██████████| 219/219 [00:19<00:00, 11.02it/s]


FineTune Epoch 1 Accuracy:90.94%


100%|██████████| 219/219 [00:16<00:00, 13.53it/s]


FineTune Epoch 2 Accuracy:92.53%


100%|██████████| 219/219 [00:15<00:00, 13.72it/s]


FineTune Epoch 3 Accuracy:93.00%


100%|██████████| 219/219 [00:16<00:00, 13.65it/s]


FineTune Epoch 4 Accuracy:93.53%


100%|██████████| 219/219 [00:16<00:00, 13.56it/s]

FineTune Epoch 5 Accuracy:93.56%


In [54]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in valid_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Validation Accuracy:",100*correct/total)

Validation Accuracy: 93.95


In [55]:
correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Test Accuracy:",100*correct/total)

Test Accuracy: 94.1


In [58]:
torch.save(model.state_dict(),"gender_mobilenet_model.pth")

In [61]:
torch.save(model.state_dict(), "/kaggle/working/gender_mobilenet_model.pth")